# Online Student Engagement Detection - Model Training on Google Colab

This notebook is configured to train the **ST-GCN**, **Bi-GRU**, and **ConvNeXt1D** models on Google Colab using a free **GPU** (T4 / V100 / A100).

### Pre-requisites:
1. **Zip your processed dataset**: Zip your local `processed/` directory (containing all `.npy` files and `labels.csv`) into a single file named `processed.zip`.
2. **Upload to Google Drive**: Upload the `processed.zip` to your Google Drive in a folder named `student-engagement` (i.e. `/My Drive/student-engagement/processed.zip`).

## Step 1: Enable GPU Acceleration

Before running any cells, ensure that you have connected to a GPU runtime:
1. Go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or any available GPU) under *Hardware accelerator*.
3. Click **Save**.

## Step 2: Clone Repository & Install Dependencies

In [ ]:
# Clone your github repository
!git clone https://github.com/Shivani367/student-engagement.git
%cd student-engagement

In [ ]:
# Install requirements
!pip install -r requirements.txt
# Ensure mediapipe and pandas are installed
!pip install mediapipe pandas tqdm matplotlib scikit-learn

## Step 3: Mount Google Drive and Copy Processed Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Copy the zipped dataset from Google Drive and extract it
# Adjust the path if you uploaded processed.zip to a different directory in Drive
!cp /content/drive/MyDrive/student-engagement/processed.zip .
!unzip -q processed.zip -d .

# Verify that the processed files are extracted correctly
import os
print("Check labels.csv exists:", os.path.exists("processed/labels.csv"))
print("Number of preprocessed landmark files:", len(os.listdir("processed")) - 1)

## Step 4: Run Training on GPU

We will train all three models. Training on Colab's T4 GPU should take about **1-2 minutes per epoch**.

### 1. Train ST-GCN Model

In [ ]:
!python train_stgcn.py --epochs 15 --batch_size 32

### 2. Train Bi-GRU Model

In [ ]:
!python train_bigru.py --epochs 15 --batch_size 64

### 3. Train ConvNeXt1D Model

In [ ]:
!python train_convnext.py --epochs 15 --batch_size 32

## Step 5: Evaluate Models and Ensemble

In [ ]:
# Run evaluation script
!python evaluate.py

In [ ]:
# Display confusion matrices
import os
from IPython.display import Image, display

results_dir = 'evaluation_results'
if os.path.exists(results_dir):
    for f in os.listdir(results_dir):
        if f.endswith('_cm.png'):
            print(f"\n--- Confusion Matrix: {f} ---")
            display(Image(os.path.join(results_dir, f)))

## Step 6: Backup Checkpoints to Google Drive

This copies the trained checkpoints back to your Google Drive so that you can download them onto your laptop and load them in your Streamlit web app!

In [ ]:
# Copy checkpoints folder to Drive
!mkdir -p /content/drive/MyDrive/student-engagement/checkpoints_backup
!cp -r checkpoints/* /content/drive/MyDrive/student-engagement/checkpoints_backup/
print("Checkpoints successfully backed up to your Google Drive!")